<a href="https://colab.research.google.com/github/biopharma26/PracticeNotebooks/blob/main/4_1_NGS_Data_Formats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1: NGS Data Formats — FASTQ and SAM/BAM

**Estimated time:** 90 minutes  **Prerequisites:** none — this notebook installs everything it needs.

### Learning Objectives
- Identify and interpret the four-line structure of a FASTQ record, including Phred-encoded quality scores.
- Distinguish the header, alignment, and CIGAR components of SAM/BAM files.
- Run a basic quality-control check on raw sequencing data and interpret the output.
- Relate SAM alignment fields to how a read sits against a reference sequence.

> Run the cells top to bottom (**Runtime ▸ Run all** works too). All data used here is generated inside the notebook, so no uploads are required. A couple of cells shell out to command-line tools (`fastqc`, `samtools`) that are installed in the first code cell.

## Setup
Install the small set of tools this lab needs (~30–60 seconds).

In [ ]:
%%capture
!pip -q install biopython pysam
!apt-get -qq update
!apt-get -qq install -y fastqc samtools > /dev/null

## Background

FASTQ is the standard text format for raw sequencing reads: each read is stored as **four lines** — a header beginning with `@`, the nucleotide sequence, a `+` separator line, and an ASCII-encoded **Phred quality string**, where

$$Q = -10 \log_{10} P$$

and *P* is the probability that a given base call is wrong. SAM (Sequence Alignment/Map) stores the results of aligning reads to a reference genome as a tab-delimited text format; BAM is its binary, compressed, indexed counterpart. SAM files have a header section (lines starting with `@`, e.g. `@SQ` for the reference dictionary) followed by an alignment section with 11 mandatory fields per read — including the FLAG, reference name, position, MAPQ, and the CIGAR string describing matches, insertions, and deletions.

## Part A — Decoding a FASTQ Record

In [ ]:
fastq_lines = [
    '@SRR000001.1 read_1 length=36',
    'GATCTTGGAACAATCTGATCTGTAGGCTGGTTCAG',
    '+',
    "!''*((((***+))%%%++)(%%%%).1***-+*",
]
header, seq, plus, qual = fastq_lines
print("Header    :", header)
print("Sequence  :", seq)
print("Separator :", plus)
print("Quality   :", qual)
print("Read length:", len(seq), "| Quality string length:", len(qual))


In [ ]:
def phred_quality(qual_char, offset=33):
    # Phred+33 encoding: Q = ASCII value of the character minus 33.
    return ord(qual_char) - offset

def error_probability(q):
    return 10 ** (-q / 10)

for pos in [1, 20]:  # 1-based positions requested in the lab
    idx = pos - 1
    q_score = phred_quality(qual[idx])
    p_err = error_probability(q_score)
    print(f"Position {pos:>2}: base={seq[idx]}  quality_char={qual[idx]!r}  "
          f"Q={q_score:>2}  P(error)={p_err:.4f}")


**Q1.** Identify each of the four lines above and state what information each one encodes.

_Your answer:_



**Q2.** Using the printed Q scores, which of the two positions (1 or 20) is more reliable, and by how much (compare P(error))?

_Your answer:_



In [ ]:
import matplotlib.pyplot as plt

q_scores = [phred_quality(c) for c in qual]
plt.figure(figsize=(8, 3))
plt.plot(range(1, len(q_scores) + 1), q_scores, marker="o")
plt.axhline(20, color="red", linestyle="--", linewidth=1, label="Q=20 (1% error) threshold")
plt.xlabel("Read position")
plt.ylabel("Phred quality score (Q)")
plt.title("Per-base quality — single FASTQ record")
plt.legend()
plt.tight_layout()
plt.show()


**Q3.** Which region of this read is the least reliable, and why might that matter during downstream analysis?

_Your answer:_



**Q4.** Explain, in your own words, why adapter trimming and quality filtering are typically performed before alignment rather than after.

_Your answer:_



## Part B — Simulating Reads and Running Quality Control
A single record does not tell you much about a whole run. Here we generate a small synthetic FASTQ file with 3'-end quality decay and occasional adapter read-through, then run **FastQC** on it, exactly as you would on real sequencer output.

In [ ]:
import random
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import Bio.SeqIO as SeqIO

random.seed(42)
BASES = "ACGT"
ADAPTER = "AGATCGGAAGAGC"
READ_LEN = 75
N_READS = 300

def simulate_quality(length):
    quals = []
    for i in range(length):
        # quality decays quadratically toward the 3' end, like real Illumina runs
        decay = int(20 * (i / length) ** 2)
        q = max(38 - decay + random.randint(-3, 3), 2)
        quals.append(q)
    return quals

records = []
for i in range(N_READS):
    seq = "".join(random.choice(BASES) for _ in range(READ_LEN))
    if random.random() < 0.15:  # simulate adapter read-through in ~15% of reads
        cut = random.randint(45, 65)
        seq = (seq[:cut] + ADAPTER * 6)[:READ_LEN]  # repeat adapter to guarantee full read length
    rec = SeqRecord(Seq(seq), id=f"sim_read_{i}", description="")
    rec.letter_annotations["phred_quality"] = simulate_quality(READ_LEN)
    records.append(rec)

with open("reads.fastq", "w") as f:
    SeqIO.write(records, f, "fastq")

print(f"Wrote {len(records)} simulated reads to reads.fastq")


In [ ]:
!fastqc reads.fastq -o . -q
!unzip -o -q reads_fastqc.zip


In [ ]:
from IPython.display import Image, display
display(Image(filename="reads_fastqc/Images/per_base_quality.png"))


In [ ]:
# Peek at the "Overrepresented sequences" module of the FastQC report
!sed -n '/>>Overrepresented sequences/,/>>END_MODULE/p' reads_fastqc/fastqc_data.txt | head -20


**Q5.** Does the per-base quality plot show the expected drop toward the 3' end? At roughly what position does mean quality fall below Q20?

_Your answer:_



**Q6.** Does FastQC flag the injected adapter sequence as overrepresented? Based on this, recommend one concrete pre-processing step and justify it.

_Your answer:_



## Part C — Reading a SAM Record
To see SAM/BAM fields in context we build a miniature reference and align our Part A read to it — small enough to inspect by eye, but produced with the real `samtools` toolchain.

In [ ]:
# A tiny reference: 10bp of flanking sequence + the 36bp read from Part A, so the
# read aligns perfectly at position 11 with CIGAR "36M".
reference = "ACGTACGTAC" + seq
with open("ref.fasta", "w") as f:
    f.write(">chrTest\n" + reference + "\n")

!samtools faidx ref.fasta
print("Reference length:", len(reference))


In [ ]:
sam_header = f"@HD\tVN:1.6\tSO:coordinate\n@SQ\tSN:chrTest\tLN:{len(reference)}\n"
sam_record = f"SRR000001.1\t0\tchrTest\t11\t60\t36M\t*\t0\t0\t{seq}\t{qual}\n"

with open("aln.sam", "w") as f:
    f.write(sam_header + sam_record)

!samtools view -bT ref.fasta aln.sam > aln.bam
!samtools sort aln.bam -o aln.sorted.bam
!samtools index aln.sorted.bam
!samtools view -h aln.sorted.bam


In [ ]:
import pysam

bam = pysam.AlignmentFile("aln.sorted.bam", "rb")
for read in bam:
    print("Query name :", read.query_name)
    print("Flag       :", read.flag, "(0 = mapped, forward strand)")
    print("Reference  :", read.reference_name)
    print("Position   :", read.reference_start + 1, "(1-based)")
    print("MAPQ       :", read.mapping_quality)
    print("CIGAR      :", read.cigarstring)
bam.close()


**Q7.** Label each of the 11 mandatory SAM fields shown in the raw `samtools view` output above.

_Your answer:_



**Q8.** The CIGAR string here is '36M'. What would '30M2I4M' mean instead?

_Your answer:_



In [ ]:
# A simple text pileup, as a stand-in for a genome-browser (IGV) screenshot
print("Reference:", reference)
print("Read     :", " " * 10 + seq)
print(" " * 10 + "^-- read starts at position 11 (matches SAM POS field)")


**Optional — real IGV view.** If you'd rather see this in an interactive genome browser widget, uncomment and run the cell below (requires the `igv-notebook` package; behavior can vary by Colab environment).

In [ ]:
# !pip -q install igv-notebook
# import igv_notebook
# igv_notebook.init()
# b = igv_notebook.Browser({
#     "genome": "ASM584v2",  # placeholder genome; for a fully custom reference use a "reference" dict
#     "locus": "chrTest:1-66",
# })


**Q9.** Explain the meaning of the MAPQ value of 60 printed above, and how MAPQ influences confidence in variant calls made later.

_Your answer:_



## Discussion Questions

**Q10.** Why is BAM used for storage and analysis instead of SAM, even though they contain the same information?

_Your answer:_



**Q11.** What could cause a read to have a FLAG value indicating it is unmapped, and how would that appear in a SAM file?

_Your answer:_



**Q12.** How would low base-quality scores in a specific region of the genome affect variant calling confidence in that region?

_Your answer:_



---
### Deliverable
Save this notebook (**File ▸ Download ▸ .ipynb**, or File ▸ Save a copy in Drive) with all cells run and all answer cells filled in, along with the per-base quality plot and the FastQC overrepresented-sequences output.